# NASA baseline comparison

This notebook recomputes the manuscript NASA benchmark from the stored seed-42 predictions. It does not retrain any model.

Held-out test set: B0018 (132 cycles, 24 °C), B0032 (39 cycles, 43 °C), and the B0053 test portion (16 cycles, 4 °C). Total: 187 cycles.

Macro RMSE, MAE, MAPE, and $R^2$ are unweighted means of the three cell-level metrics. MaxE is the maximum absolute error over all 187 cycles. CALCE results are not included.


In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "nasa_manuscript" / "nasa_heldout_predictions.csv").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the TE-Q-Transformer repository.")

ROOT = repo_root()
PRED_PATH = ROOT / "data" / "nasa_manuscript" / "nasa_heldout_predictions.csv"
print("Repository root:", ROOT)
print("Prediction file:", PRED_PATH)


Repository root: E:\TE-Q-Transformer\TE-Q-Transformer
Prediction file: E:\TE-Q-Transformer\TE-Q-Transformer\data\nasa_manuscript\nasa_heldout_predictions.csv


## Split check and metrics


In [2]:
df = pd.read_csv(PRED_PATH)
expected = {"B0018": 132, "B0032": 39, "B0053_test": 16}
order = ["LSTM", "GRU", "CNN1D", "TCN", "DLinear", "Transformer", "PatchTST", "iTransformer", "QLSTM", "QNN-GRU", "TE-Q-Transformer"]
if set(df["model"]) != set(order):
    raise RuntimeError(f"Unexpected models: {sorted(df['model'].unique())}")
if sorted(df["seed"].unique()) != [42]:
    raise RuntimeError(f"Unexpected seeds: {df['seed'].unique()}")

def cell_metrics(y, pred):
    error = pred - y
    rmse = float(np.sqrt(np.mean(error ** 2)))
    mae = float(np.mean(np.abs(error)))
    mape = float(np.mean(np.abs(error) / np.maximum(np.abs(y), 1e-8)) * 100.0)
    ss_res = float(np.sum(error ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r2 = float(1.0 - ss_res / ss_tot)
    return rmse, mae, mape, r2

rows = []
for model in order:
    part = df[df["model"] == model]
    counts = part.groupby("cell").size().to_dict()
    if counts != expected:
        raise RuntimeError(f"{model} split {counts}")
    per_cell = []
    for cell in ["B0018", "B0032", "B0053_test"]:
        group = part[part["cell"] == cell]
        per_cell.append(cell_metrics(group["true_soh"].to_numpy(), group["pred_soh"].to_numpy()))
    macro = np.mean(per_cell, axis=0)
    maxe = float(np.max(np.abs(part["pred_soh"] - part["true_soh"])))
    rows.append({
        "Model": model,
        "RMSE": macro[0],
        "MAE": macro[1],
        "MAPE (%)": macro[2],
        "R2": macro[3],
        "MaxE": maxe,
        "N": int(len(part)),
    })
table = pd.DataFrame(rows)
print("Total prediction rows:", len(df), "Models x 187 =", 11 * 187)
print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))


Total prediction rows: 2057 Models x 187 = 2057


           Model   RMSE    MAE  MAPE (%)      R2   MaxE   N
            LSTM 0.0400 0.0348    3.7919  0.0512 0.0840 187
             GRU 0.0444 0.0384    4.2119 -0.4143 0.0926 187
           CNN1D 0.0509 0.0457    4.7807 -3.1912 0.1245 187
             TCN 0.0535 0.0452    5.0892 -0.3410 0.1491 187
         DLinear 0.0624 0.0568    5.8847 -2.2316 0.1853 187
     Transformer 0.0452 0.0400    4.4538 -0.0165 0.1119 187
        PatchTST 0.0387 0.0333    3.6878  0.3211 0.1457 187
    iTransformer 0.0341 0.0298    3.2830  0.5328 0.1165 187
           QLSTM 0.0445 0.0386    4.3496 -0.3360 0.1066 187
         QNN-GRU 0.0304 0.0248    2.6102  0.2579 0.0725 187
TE-Q-Transformer 0.0168 0.0141    1.5604  0.8698 0.1022 187


## Per-cell RMSE


In [3]:
records = []
for model in order:
    part = df[df["model"] == model]
    row = {"Model": model}
    for cell in ["B0018", "B0032", "B0053_test"]:
        group = part[part["cell"] == cell]
        error = group["pred_soh"].to_numpy() - group["true_soh"].to_numpy()
        row[cell] = float(np.sqrt(np.mean(error ** 2)))
    records.append(row)
per_cell = pd.DataFrame(records)
print(per_cell.to_string(index=False, float_format=lambda v: f"{v:.4f}"))


           Model  B0018  B0032  B0053_test
            LSTM 0.0505 0.0370      0.0324
             GRU 0.0499 0.0395      0.0438
           CNN1D 0.0255 0.0385      0.0886
             TCN 0.0809 0.0469      0.0326
         DLinear 0.0503 0.1085      0.0284
     Transformer 0.0676 0.0360      0.0320
        PatchTST 0.0679 0.0215      0.0268
    iTransformer 0.0584 0.0285      0.0154
           QLSTM 0.0609 0.0276      0.0450
         QNN-GRU 0.0275 0.0330      0.0306
TE-Q-Transformer 0.0271 0.0119      0.0113


## What this table is

TE-Q-Transformer is the primary reference model. The other ten rows are the baseline models evaluated on the same 187 cycles. The numbers are calculated from `data/nasa_manuscript/nasa_heldout_predictions.csv`. This notebook does not refit scalers, change the split, or include the NASA-to-CALCE transfer run.
